# Notebook 23 — WU20D full-native-volume nnU-Net qualification

**CPU only; no training, no learned model inference, and no held-out lesion labels.** Process four prespecified problematic held-out ISLES2024 scans using their own fold-specific training-only parameters. Verify that real full-volume MTT NaN/Inf values are accounted for, and compare actual nnU-Net 2.8.1 preprocessed 506/507 tensors. Original Dataset504/505 remain frozen. The anatomical support remains a **provisional head-tissue proxy**, not an approved brain mask.

The notebook runs one patient per independent process and saves only resumable private JSON audit results. Full NIfTI intermediates are temporary.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Verify frozen sources and pin the exact audited code

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, json, os, subprocess, sys
ROOT=Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RAW=ROOT/'nnUNet_raw'/'Dataset505_ISLES2024_NCCT_PERFUSION'
PP=ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLITS=PP/'splits_final.json'
PLANS=PP/'nnUNetPlans.json'
FOLDS=ROOT/'results'/'wu20b_full_fold_calibration'/'fold_private'
OUTPUT=ROOT/'results'/'wu20d_full_native_volume_qualification'
SOURCE=Path('/content/wu20d_pinned_source')
PKG=SOURCE/'src'/'hybridstrokeseg'
PKG.mkdir(parents=True,exist_ok=True)
(SOURCE/'scripts').mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1<<20),b''):h.update(block)
    return h.hexdigest()
assert RAW.is_dir() and SPLITS.is_file() and PLANS.is_file()
assert sha256(SPLITS)=='664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
assert sha256(PLANS)=='6203521136de2f57bbe3895a639863112a5869697de9b752ade42dc1a23e4dca'
CASES={'sub_stroke0016':2, 'sub_stroke0113':1, 'sub_stroke0137':4, 'sub_stroke0152':3}
for case,fold in CASES.items():
    assert (FOLDS/f'WU20B_fold{fold}_parameters_PRIVATE.json').is_file()
    for ch in range(5):
        assert (RAW/'imagesTr'/f'{case}_{ch:04d}.nii.gz').is_file()
# Immutable Git commit containing the audited script and its dependency versions.
COMMIT='c887722161f1370add73bdddeac75610894021d8'
BASE=f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{COMMIT}'
SOURCES={
 'src/hybridstrokeseg/wu19_normalization_protocol.py':PKG/'wu19_normalization_protocol.py',
 'src/hybridstrokeseg/wu20_fold_preprocessing.py':PKG/'wu20_fold_preprocessing.py',
 'scripts/stage_isles2024_fold_pilot.py':SOURCE/'scripts'/'stage_isles2024_fold_pilot.py',
 'scripts/qualify_isles2024_nnunet_runtime.py':SOURCE/'scripts'/'qualify_isles2024_nnunet_runtime.py',
 'scripts/qualify_isles2024_full_volume.py':SOURCE/'scripts'/'qualify_isles2024_full_volume.py',
}
for relative,dest in SOURCES.items():
    subprocess.run(['curl','-fLsS','--retry','5',f'{BASE}/{relative}','-o',str(dest)],check=True)
    assert dest.stat().st_size>500,f'Missing/truncated: {relative}'
    print('Pinned source',relative,sha256(dest)[:14])
print('Frozen splits and plans: VERIFIED')


## Install exact nnU-Net 2.8.1 (CPU) and verify normalization

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','nnunetv2==2.8.1','nibabel>=5.2,<6'],check=True)
assert importlib.metadata.version('nnunetv2')=='2.8.1'
print('Verified nnU-Net',importlib.metadata.version('nnunetv2'))


## Run four full-native-volume held-out cases sequentially

No held-out lesion labels or prediction outcomes are read. Each case is a separate Python process and its private result is written atomically to Google Drive. **If a case fails, stop and inspect the log—do not weaken any checks.** Existing completed case records are reused only when all five original NIfTI SHA256 values and the corresponding fold manifest are unchanged.

In [ ]:
OUTPUT.mkdir(parents=True,exist_ok=True)
LOG=OUTPUT/'WU20D_full_native_volume_CPU.log'
env=os.environ.copy()
env['PYTHONPATH']=str(SOURCE/'src')+os.pathsep+str(SOURCE/'scripts')+os.pathsep+env.get('PYTHONPATH','')
SCRIPT=SOURCE/'scripts'/'qualify_isles2024_full_volume.py'
for case in CASES:
    command=[sys.executable,'-u',str(SCRIPT),'--raw505',str(RAW),'--splits',str(SPLITS),
             '--source-plans',str(PLANS),'--fold-manifest-dir',str(FOLDS),
             '--output',str(OUTPUT),'--case',case]
    print('FULL-VOLUME CPU audit:',case,flush=True)
    with LOG.open('a',encoding='utf8') as stream:
        stream.write(f'\n[WU20D case] {case} pinned={COMMIT}\n')
        stream.flush()
        with subprocess.Popen(command,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True)
                stream.write(line)
                stream.flush()
            code=process.wait()
    if code:
        raise RuntimeError(f'{case} failed (exit {code}). Inspect {LOG}; training remains blocked.')
print('All four technical CPU processes finished')


## Verify case coverage and produce aggregate status (not training approval)

In [ ]:
reports=[]
for case,fold in CASES.items():
    path=OUTPUT/'private_cases'/f'{case}_full_volume_PRIVATE.json'
    assert path.is_file(),f'Missing completed private report: {case}'
    q=json.loads(path.read_text(encoding='utf8'))
    assert q['case']==case and q['fold']==fold and q['full_native_volume_tested'] is True
    assert q['training_ready'] is False and q['heldout_labels_read'] is False
    assert q['real_model_inference_performed'] is False
    assert q['status'] in ('FULL_VOLUME_TECHNICAL_PASS','FULL_VOLUME_TECHNICAL_REVIEW_REQUIRED')
    reports.append(q)
agg={
 'schema':'WU20D-four-case-aggregate-1.0',
 'patients_qualified':len(reports),
 'case_status':{x['case']:x['status'] for x in reports},
 'per_case_raw_mtt_nonfinite':{x['case']:x['raw_channel_qc']['MTT']['n_nonfinite_full'] for x in reports},
 'per_case_mtt_nonfinite_inside_support':{x['case']:x['raw_channel_qc']['MTT']['n_nonfinite_within_support'] for x in reports},
 'per_case_postresample_overshoot_voxels':{x['case']:x['official_preprocessing']['postresample_perfusion_voxels_abs_gt_8'] for x in reports},
 'training_ready':False,
 'independent_anatomical_review_pending':True,
 'real_model_inference_performed':False,
 'dataset504_505_unchanged':True,
}
(OUTPUT/'WU20D_four_case_aggregate_PRIVATE.json').write_text(json.dumps(agg,indent=2,sort_keys=True)+'\n',encoding='utf8')
print(json.dumps(agg,indent=2))
